In [1]:
from pyspark.sql import SparkSession

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

import pandas as pd

Create Spark Session

In [2]:
spark = (
    SparkSession.builder
    .appName("Random Forest - Partial Oversampling")
    .master("local[*]")
    .getOrCreate()
)

26/07/30 23:56:47 WARN Utils: Your hostname, LAPTOP-S89A4J3G resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/30 23:56:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/30 23:56:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.csv(
    "../cleaned_data/road_accident_cleaned",
    header=True,
    inferSchema=True
)

In [4]:
print("Rows :", df.count())
print("Columns :", len(df.columns))

df.printSchema()

Rows : 300491
Columns : 23
root
 |-- Accident_Index: string (nullable = true)
 |-- Accident Date: date (nullable = true)
 |-- Month: string (nullable = true)
 |-- Day_of_Week: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Junction_Control: string (nullable = true)
 |-- Junction_Detail: string (nullable = true)
 |-- Accident_Severity: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Light_Conditions: string (nullable = true)
 |-- Local_Authority_(District): string (nullable = true)
 |-- Carriageway_Hazards: string (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Number_of_Casualties: integer (nullable = true)
 |-- Number_of_Vehicles: integer (nullable = true)
 |-- Police_Force: string (nullable = true)
 |-- Road_Surface_Conditions: string (nullable = true)
 |-- Road_Type: string (nullable = true)
 |-- Speed_limit: integer (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Urban_or_Rural_Area: string (nullable = true)
 |

Select Features

In [5]:
feature_cols = [
    "Month",
    "Day_of_Week",
    "Junction_Control",
    "Junction_Detail",
    "Light_Conditions",
    "Carriageway_Hazards",
    "Number_of_Casualties",
    "Number_of_Vehicles",
    "Road_Surface_Conditions",
    "Road_Type",
    "Speed_limit",
    "Urban_or_Rural_Area",
    "Weather_Conditions",
    "Vehicle_Type"
]

target_col = "Accident_Severity"

Create Label Indexer

In [6]:
label_indexer = StringIndexer(
    inputCol=target_col,
    outputCol="label"
)

In [7]:
categorical_cols = [
    "Month",
    "Day_of_Week",
    "Junction_Control",
    "Junction_Detail",
    "Light_Conditions",
    "Carriageway_Hazards",
    "Road_Surface_Conditions",
    "Road_Type",
    "Urban_or_Rural_Area",
    "Weather_Conditions",
    "Vehicle_Type"
]

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_Index",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=c + "_Index",
        outputCol=c + "_Vec"
    )
    for c in categorical_cols
]

Assemble Features

In [8]:
assembler_inputs = [
    c + "_Vec" for c in categorical_cols
] + [
    "Number_of_Casualties",
    "Number_of_Vehicles",
    "Speed_limit"
]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

Build Pipeline

In [9]:
pipeline = Pipeline(
    stages=[label_indexer] +
           indexers +
           encoders +
           [assembler]
)

pipeline_model = pipeline.fit(df)

processed_df = pipeline_model.transform(df)

Train/Test Split

In [10]:
train_df, test_df = processed_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

In [11]:
train_df.groupBy("label") \
    .count() \
    .orderBy("label") \
    .show()

26/07/30 23:57:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/07/30 23:57:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 23:57:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 23:57:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 23:57:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+-----+------+
|label| count|
+-----+------+
|  0.0|205068|
|  1.0| 32134|
|  2.0|  3081|
+-----+------+



In [12]:
slight = train_df.filter("label = 0")
serious = train_df.filter("label = 1")
fatal = train_df.filter("label = 2")

In [13]:
slight_count = slight.count()
serious_count = serious.count()
fatal_count = fatal.count()

print("Slight :", slight_count)
print("Serious:", serious_count)
print("Fatal  :", fatal_count)

Slight : 205068
Serious: 32134
Fatal  : 3081


Partial Oversampling Targets

In [14]:
target_serious = 100000
target_fatal = 50000

serious_fraction = target_serious / serious_count
fatal_fraction = target_fatal / fatal_count

print("Serious Fraction :", serious_fraction)
print("Fatal Fraction :", fatal_fraction)

Serious Fraction : 3.111968631356196
Fatal Fraction : 16.22849724115547


In [15]:
serious_over = serious.sample(
    withReplacement=True,
    fraction=serious_fraction,
    seed=42
)

fatal_over = fatal.sample(
    withReplacement=True,
    fraction=fatal_fraction,
    seed=42
)

Create New Training Dataset

In [16]:
balanced_train = (
    slight
    .unionByName(serious_over)
    .unionByName(fatal_over)
)

In [17]:
balanced_train.groupBy("label") \
    .count() \
    .orderBy("label") \
    .show()

26/07/31 00:01:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/31 00:01:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/31 00:01:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/31 00:01:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/31 00:01:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/31 00:01:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/31 00:02:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/31 00:02:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/31 00:02:05 WARN RowBasedKeyValueBatch: Calling spill() on

+-----+------+
|label| count|
+-----+------+
|  0.0|205068|
|  1.0| 99738|
|  2.0| 50186|
+-----+------+



Train Random Forest

In [18]:
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(balanced_train)

26/07/31 00:03:11 WARN MemoryStore: Not enough space to cache rdd_181_5 in memory! (computed 12.3 MiB so far)
26/07/31 00:03:11 WARN BlockManager: Persisting block rdd_181_5 to disk instead.
26/07/31 00:03:42 WARN DAGScheduler: Broadcasting large task binary with size 1210.5 KiB
26/07/31 00:03:52 WARN DAGScheduler: Broadcasting large task binary with size 1734.2 KiB
26/07/31 00:04:08 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/07/31 00:04:23 WARN DAGScheduler: Broadcasting large task binary with size 4.4 MiB
26/07/31 00:04:46 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB
26/07/31 00:05:12 WARN DAGScheduler: Broadcasting large task binary with size 1583.8 KiB
26/07/31 00:05:14 WARN DAGScheduler: Broadcasting large task binary with size 11.8 MiB
26/07/31 00:05:46 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB


In [19]:
predictions = rf_model.transform(test_df)

In [20]:
accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(predictions)

print("Accuracy :", accuracy)

26/07/31 00:06:26 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB


Accuracy : 0.8453527770395961


In [21]:
precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(predictions)

print("Precision :", precision)

26/07/31 00:07:00 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB


Precision : 0.7810383940243086


In [23]:
recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(predictions)

print("Recall :", recall)

26/07/31 00:07:47 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB


Recall : 0.8453527770395961


In [24]:
f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions)

print("F1 Score :", f1)

26/07/31 00:08:02 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB


F1 Score : 0.7891099026617476


In [25]:
predictions.groupBy(
    "label",
    "prediction"
).count().orderBy(
    "label",
    "prediction"
).show(50)

26/07/31 00:08:16 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB
26/07/31 00:08:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/31 00:08:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/31 00:08:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/31 00:08:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/31 00:08:28 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|50722|
|  0.0|       1.0|  199|
|  0.0|       2.0|  527|
|  1.0|       0.0| 7591|
|  1.0|       1.0|  113|
|  1.0|       2.0|  245|
|  2.0|       0.0|  737|
|  2.0|       1.0|   12|
|  2.0|       2.0|   62|
+-----+----------+-----+

